In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from fedbiomed.common.training_plans import TorchTrainingPlan
from fedbiomed.common.data import DataManager
import pandas as pd
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Dataset personalizado que carga CSV desde la ruta local del nodo
class BreastCancerDataset(Dataset):
    def __init__(self, csv_path):
        df = pd.read_csv(csv_path)
        # Asumiendo que las columnas de características están en df.columns[:-1]
        # y la última columna es la etiqueta
        X = df.iloc[:, :-1].values.astype(np.float32)
        y = df.iloc[:, -1].values.astype(np.int64)

        # Normalizar características
        scaler = StandardScaler()
        X = scaler.fit_transform(X)

        self.X = torch.tensor(X)
        self.y = torch.tensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class MyTrainingPlan(TorchTrainingPlan):

    class BreastCancerDataset(torch.utils.data.Dataset):
        def __init__(self, csv_path):
            import pandas as pd
            from sklearn.preprocessing import StandardScaler
            df = pd.read_csv(csv_path)
            X = df.iloc[:, :-1].values.astype(np.float32)
            y = df.iloc[:, -1].values.astype(np.int64)
            scaler = StandardScaler()
            X = scaler.fit_transform(X)
            self.X = torch.tensor(X)
            self.y = torch.tensor(y)

        def __len__(self):
            return len(self.y)

        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]

    def init_model(self, model_args):
        class Net(nn.Module):
            def __init__(self):
                super().__init__()
                self.fc1 = nn.Linear(539, 16)
                self.fc2 = nn.Linear(16, 2)

            def forward(self, x):
                x = F.relu(self.fc1(x))
                x = self.fc2(x)
                return F.log_softmax(x, dim=1)
        return Net()

    def init_optimizer(self, optimizer_args):
        return torch.optim.Adam(self.model().parameters(), lr=optimizer_args['lr'])

    def init_dependencies(self):
        return [
            "import numpy as np",
            'import pandas as pd',
            'from sklearn.preprocessing import StandardScaler',
            'import torch',
            'import torch.nn.functional as F'
        ]

    def training_data(self):
        # Usa la clase anidada
        dataset = self.BreastCancerDataset(self.dataset_path)
        return DataManager(dataset=dataset, shuffle=True)

    def training_step(self, data, target):
        output = self.model().forward(data)
        loss = F.nll_loss(output, target)
        return loss

In [ ]:
from fedbiomed.common.metrics import MetricTypes

model_args = {}

training_args = {
    'loader_args': {'batch_size': 32},
    'optimizer_args': {'lr': 1e-3},
    'epochs': 5,
    'dry_run': False,
    'batch_maxnum': 100,
    # otros parámetros opcionales
}

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage

tags = ['breast', 'cancer','mamography']  # los tags con los que etiquetaste tu dataset en los nodos
rounds = 2

exp = Experiment(
    tags=tags,
    model_args=model_args,
    training_plan_class=MyTrainingPlan,
    training_args=training_args,
    round_limit=rounds,
    aggregator=FedAverage(),
    tensorboard=True
)

exp.run()  # corre el entrenamiento federado

In [ ]:
try:
    exp.training_plan().export_model('./trained_model')
except Exception as e:
    print(e)

print("Training rounds keys:", exp.training_replies().keys())
print("Aggregated params keys:", exp.aggregated_params().keys())